Cleaning up HUC12 features for study area (fixing self intersections, gaps, unclosed polygons, etc)

In [ ]:
import arcpy

input_fc = r"K:\WRD\WMB_GIS\Basin_Project\StreamClassCommons.gdb\HUC12_EntireStudyArea"
output_fc = r"K:\WRD\WMB_GIS\Basin_Project\StreamClassCommons.gdb\HUC12_EntireStudyArea_cleaned"
temp_fc = r"K:\WRD\WMB_GIS\Basin_Project\StreamClassCommons.gdb\HUC12_EntireStudyArea_tmp"

# 1. Copy the dataset so you keep your original untouched
arcpy.management.CopyFeatures(input_fc, output_fc)

# 2. Repair geometries
arcpy.management.RepairGeometry(output_fc, "DELETE_NULL")

# 3. Eliminate tiny slivers (optional)
# Syntax: EliminatePolygonPart(in_features, out_feature_class, condition, part_area, part_option)
arcpy.management.EliminatePolygonPart(
    in_features=output_fc,
    out_feature_class=temp_fc,
    condition="AREA",
    part_area="0.0001 SquareKilometers",
    part_option="CONTAINED_ONLY"
)

# Replace cleaned layer with the result
arcpy.management.Delete(output_fc)
arcpy.management.Rename(temp_fc, output_fc)



Cleaning up old run of HUC12s by deleting them as seperate features and as a merged single feature

In [ ]:
import arcpy
import os

# Path to your local working geodatabase
gdb = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb"
arcpy.env.workspace = gdb

print(f"Cleaning up old feature classes in: {gdb}")

# List all feature classes
feature_classes = arcpy.ListFeatureClasses()

deleted = []
skipped = []

for fc in feature_classes:
    if fc.startswith("Catch_") or fc == "Catchments_All":
        print(f"Deleting: {fc}")
        arcpy.management.Delete(fc)
        deleted.append(fc)
    else:
        skipped.append(fc)

print("\n✅ Cleanup complete.")
print(f"Deleted {len(deleted)} feature classes.")
print(f"Remaining (kept): {len(skipped)}")


Creates dissolved upstream basins for study area HUC12s, skips dissolve if only 1 HUC

In [ ]:
import arcpy
import os
import time
import traceback
from datetime import datetime
from tqdm import tqdm

# ============================================================
# USER INPUTS
# ============================================================
gdb_path = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb"
huc_fc = os.path.join(gdb_path, "HUC12_EntireStudyArea_cleaned")
log_file = os.path.join(os.path.dirname(gdb_path), "CatchmentDissolve_Log.txt")

# ============================================================
# ENVIRONMENT
# ============================================================
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(5070)  # EPSG:5070

# ============================================================
# LOGGING FUNCTIONS
# ============================================================
def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(log_file, "a") as f:
        f.write(line + "\n")

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def load_network():
    """Read HUC12 -> ToHUC mapping into a dictionary."""
    network = {}
    with arcpy.da.SearchCursor(huc_fc, ["HUC12", "ToHUC"]) as cursor:
        for huc, tohuc in cursor:
            if huc not in network:
                network[huc] = tohuc
    return network

def find_upstream(huc, network):
    """Recursively find all upstream HUC12s that drain into the given HUC."""
    upstream = set()
    inverse = {}  # ToHUC → list of HUC12s flowing into it

    for h, toh in network.items():
        if toh not in inverse:
            inverse[toh] = []
        inverse[toh].append(h)

    def recurse(target):
        if target in inverse:
            for up in inverse[target]:
                if up not in upstream:
                    upstream.add(up)
                    recurse(up)

    recurse(huc)
    upstream.add(huc)
    return upstream

def get_completed(log_path):
    """Read completed HUC12s from the log file."""
    done = set()
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            for line in f:
                if "SUCCESS:" in line or "Single HUC" in line:
                    parts = line.split()
                    for p in parts:
                        if p.isdigit() and len(p) == 12:
                            done.add(p)
    return done

# ============================================================
# MAIN PROCESS
# ============================================================
def main():
    start_time = time.time()
    log("=== Upstream Dissolve Builder (Full Run) ===")

    network = load_network()
    all_hucs = list(network.keys())
    completed = get_completed(log_file)

    log(f"Found {len(all_hucs)} total HUC12s.")
    log(f"Resuming from {len(completed)} already completed.")

    processed = 0
    skipped = 0
    errors = 0

    with tqdm(total=len(all_hucs), desc="Processing HUC12s", ncols=80) as pbar:
        for huc in all_hucs:
            pbar.update(1)

            if huc in completed:
                skipped += 1
                continue

            try:
                out_fc = os.path.join(gdb_path, f"Catch_{huc}")
                if arcpy.Exists(out_fc):
                    log(f"{huc}: Output already exists — skipping.")
                    skipped += 1
                    continue

                tohuc = network[huc]
                if tohuc == "CLOSED BASIN":
                    upstream_set = {huc}
                else:
                    upstream_set = find_upstream(huc, network)

                # Build SQL where clause safely
                quoted = ", ".join([f"'{x}'" for x in upstream_set])
                where = f"HUC12 IN ({quoted})"
                tmp_layer = "tmp_layer"
                arcpy.management.MakeFeatureLayer(huc_fc, tmp_layer, where)


                # Optimization: skip dissolve if only one HUC
                if len(upstream_set) == 1:
                    arcpy.management.CopyFeatures(tmp_layer, out_fc)
                    log(f"{huc}: Single HUC → copied directly to {out_fc}")
                else:
                    arcpy.management.Dissolve(tmp_layer, out_fc)
                    log(f"{huc}: SUCCESS: Dissolved {len(upstream_set)} polygons → {out_fc}")

                arcpy.management.Delete(tmp_layer)
                processed += 1

            except Exception:
                errors += 1
                log(f"{huc}: ERROR:\n{traceback.format_exc()}")

    elapsed = (time.time() - start_time) / 60
    log(f"=== Run complete in {elapsed:.2f} minutes ===")
    log(f"Summary — Success: {processed}, Skipped: {skipped}, Errors: {errors}")
    log("=====================================================")

# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    main()



Calculates and Adds Area_sqkm, Perimeter_km, and Compactness fields

In [ ]:
import arcpy
import os
from datetime import datetime

# ============================================================
# USER INPUTS
# ============================================================
gdb_path = r"C:\Users\CN0401\Documents\ArcGIS\Projects\StreamClassProcessing\Default.gdb"
output_name = "Last2Catchments_All_Upstream"
log_file = os.path.join(os.path.dirname(gdb_path), "CatchmentMerge_Log.txt")

# ============================================================
# ENVIRONMENT
# ============================================================
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(5070)  # EPSG:5070 (meters)

# ============================================================
# LOGGING
# ============================================================
def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(log_file, "a") as f:
        f.write(line + "\n")

# ============================================================
# MAIN
# ============================================================
def main():
    log("=== Step 2: Merging dissolved catchments ===")

    catch_fcs = [fc for fc in arcpy.ListFeatureClasses() if fc.startswith("Catch_")]
    log(f"Found {len(catch_fcs)} Catch_* feature classes to merge.")

    if not catch_fcs:
        log("No Catch_* feature classes found. Exiting.")
        return

    # --- ensure each input has an HUC12 field ---
    for fc in catch_fcs:
        fields = [f.name for f in arcpy.ListFields(fc)]
        if "HUC12" not in fields:
            arcpy.management.AddField(fc, "HUC12", "TEXT", field_length=12)
            huc_val = os.path.basename(fc).split("_")[1]
            with arcpy.da.UpdateCursor(fc, ["HUC12"]) as cur:
                for row in cur:
                    row[0] = huc_val
                    cur.updateRow(row)

    # --- merge ---
    merged_fc = os.path.join(gdb_path, output_name)
    log(f"Merging into: {merged_fc}")
    arcpy.management.Merge(catch_fcs, merged_fc)

    # --- add geometry-based fields ---
    for fld, type_, prec in [("TotalDisAreaSqKm", "DOUBLE", None),
                             ("PerimeterKm", "DOUBLE", None),
                             ("Compactness", "DOUBLE", None)]:
        if fld not in [f.name for f in arcpy.ListFields(merged_fc)]:
            arcpy.management.AddField(merged_fc, fld, type_, field_precision=prec)

    # --- calculate geometry fields ---
    log("Calculating area, perimeter, and compactness...")
    with arcpy.da.UpdateCursor(
        merged_fc, ["SHAPE@AREA", "SHAPE@LENGTH", "TotalDisAreaSqKm", "PerimeterKm", "Compactness"]
    ) as cur:
        for row in cur:
            area_m2 = row[0]
            perim_m = row[1]
            area_km2 = area_m2 / 1_000_000.0
            perim_km = perim_m / 1_000.0
            row[2] = area_km2
            row[3] = perim_km
            row[4] = (4 * 3.141592653589793 * area_m2) / (perim_m ** 2) if perim_m > 0 else None
            cur.updateRow(row)

    log(f"✅ Merge complete. Created: {merged_fc}")
    log("=== Step 2 finished successfully ===")

if __name__ == "__main__":
    main()





Simplified script to fill in perimeter, area, and compactness values for HUCs I mistakenly left out of the first round and which I am including now. 

In [ ]:
import arcpy

fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\StreamClassProcessing\Default.gdb\Final2HUC12_ForRasterData"

arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(5070)  # EPSG:5070 meters

# Geometry fields we expect
fields_needed = ["TotalDisAreaSqKm", "PerimeterKm", "Compactness"]

# Ensure all fields exist
existing = [f.name for f in arcpy.ListFields(fc)]
for fld in fields_needed:
    if fld not in existing:
        arcpy.AddField_management(fc, fld, "DOUBLE")
        print(f"✅ Added missing field: {fld}")
    else:
        print(f"ℹ️ Field exists: {fld} (values will be overwritten)")

print("\n📐 Calculating Area, Perimeter, Compactness...")

with arcpy.da.UpdateCursor(
    fc, ["SHAPE@AREA", "SHAPE@LENGTH", "TotalDisAreaSqKm", "PerimeterKm", "Compactness"]
) as cur:
    for row in cur:
        area_m2 = row[0]
        perim_m = row[1]

        area_km2 = area_m2 / 1_000_000.0
        perim_km = perim_m / 1_000.0

        compact = (4 * 3.141592653589793 * area_m2) / (perim_m ** 2) if perim_m > 0 else None

        row[2] = area_km2
        row[3] = perim_km
        row[4] = compact

        cur.updateRow(row)

print("\n✅ Geometry values updated successfully")


This combines all the polygon feature classes within the Catchments dataset into a single feature class

In [ ]:
import arcpy
import os
from datetime import datetime
from tqdm import tqdm

# === USER INPUTS ===
workspace_gdb = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb"
merged_name = "Catchments_All"  # name of merged output
log_file = os.path.join(os.path.dirname(workspace_gdb), "CatchmentMerge_Log.txt")

# === SETUP ===
arcpy.env.workspace = workspace_gdb
arcpy.env.overwriteOutput = True

def log(msg):
    """Simple logging utility"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(log_file, "a") as f:
        f.write(line + "\n")

# === MAIN MERGE PROCESS ===
def main():
    log("=== Catchment Merge Started ===")

    # 1️⃣ List all Catch_ feature classes
    fcs = arcpy.ListFeatureClasses("Catch_*")
    log(f"Found {len(fcs)} Catch_ feature classes to merge.")

    if not fcs:
        log("No feature classes found — aborting.")
        return

    # 2️⃣ Define output path
    merged_fc = os.path.join(workspace_gdb, merged_name)

    # 3️⃣ Delete any existing output
    if arcpy.Exists(merged_fc):
        log(f"Existing {merged_fc} found — deleting it first.")
        arcpy.Delete_management(merged_fc)

    # 4️⃣ Merge all Catch_ feature classes
    log("Merging catchments — this might take a while...")
    with tqdm(total=len(fcs), desc="Merging Catchments", ncols=80) as pbar:
        # ArcPy’s Merge can take a long time; use temporary batching if needed
        batch_size = 1000
        temp_batches = []
        for i in range(0, len(fcs), batch_size):
            batch = fcs[i:i+batch_size]
            temp_out = os.path.join(workspace_gdb, f"_temp_merge_{i//batch_size}")
            arcpy.management.Merge(batch, temp_out)
            temp_batches.append(temp_out)
            pbar.update(len(batch))

        # Merge all batches into the final output
        if len(temp_batches) > 1:
            arcpy.management.Merge(temp_batches, merged_fc)
            for t in temp_batches:
                arcpy.Delete_management(t)
        else:
            arcpy.management.CopyFeatures(temp_batches[0], merged_fc)
            arcpy.Delete_management(temp_batches[0])

    log(f"✅ Merge complete. Output: {merged_fc}")
    log("=== Catchment Merge Finished ===")

if __name__ == "__main__":
    main()



Re-projecting rasters (which were in WGS 84) to 5070 (Albers contiguous)

In [ ]:
import arcpy, os

# Input and output folders
in_folder = r"C:\StreamClassRasters"
out_folder = r"C:\StreamClassRasters_5070"

arcpy.env.workspace = in_folder
arcpy.env.overwriteOutput = False

# Target projection: EPSG 5070
out_cs = arcpy.SpatialReference(5070)

# Create output folder if missing
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

rasters = arcpy.ListRasters("*.tif")

print(f"Found {len(rasters)} rasters to reproject")

for r in rasters:
    in_raster = os.path.join(in_folder, r)
    out_raster = os.path.join(out_folder, r)

    if os.path.exists(out_raster):
        print(f"✔ Already exists, skipping: {r}")
        continue

    print(f"→ Reprojecting: {r}")

    # Project raster
    arcpy.management.ProjectRaster(
        in_raster,
        out_raster,
        out_cs,
        resampling_type="BILINEAR"
    )

print("✅ Reprojection complete!")


Projection double check (should all say Albers 5070)

In [ ]:
import arcpy, os

folder = r"C:\StreamClassRasters_5070"
arcpy.env.workspace = folder

for r in arcpy.ListRasters("*.tif"):
    sr = arcpy.Describe(r).spatialReference
    print(f"{r} → {sr.name}")


Takes the 6,275 dissolved HUC12 catchments created above and calculates zonal statistics (mean) for all 17 rasters but elevation (which will have median returned)

In [ ]:
import arcpy, os, traceback, re

# --- Setup ---
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True   # allow overwriting
print("✅ Spatial Analyst checked out")

fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\StreamClassProcessing\Default.gdb\Final2HUC12_ForRasterData"
raster_folder = r"C:\StreamClassRasters_5070"
scratch_gdb = arcpy.env.scratchGDB

arcpy.env.workspace = raster_folder
rasters = [r for r in arcpy.ListRasters("*.tif")]

print(f"Found {len(rasters)} rasters")

for r in rasters:
    raster_path = os.path.join(raster_folder, r)
    field_name = os.path.splitext(r)[0]

    # Determine stat type
    stat_type = "MEDIAN" if "Elev" in r or "Median" in r else "MEAN"
    print(f"\n📌 {r} → Field {field_name} ({stat_type})")

    # --- Ensure field exists (or overwrite values) ---
    existing = [f.name for f in arcpy.ListFields(fc)]
    if field_name not in existing:
        arcpy.AddField_management(fc, field_name, "DOUBLE")
        print(f"✅ Field added: {field_name}")
    else:
        print(f"ℹ️ Field exists, values will be overwritten")

    # --- Create short + safe temp table name ---
    short_name = re.sub(r'[^A-Za-z0-9]', '', field_name)[:20]
    out_table = os.path.join(scratch_gdb, f"zs_{short_name}")

    # --- Delete temp table if it already exists ---
    if arcpy.Exists(out_table):
        print(f"🧹 Removing existing temp table: {out_table}")
        arcpy.Delete_management(out_table)

    # --- Run Zonal Stats ---
    try:
        arcpy.sa.ZonalStatisticsAsTable(
            fc, "HUC12", raster_path, out_table, "DATA", stat_type
        )
        print(f"✅ Zonal table created: {out_table}")
    except Exception:
        print(f"❌ ZonalStats failed for {r}")
        traceback.print_exc()
        continue

    # Field returned from ZonalStatisticsAsTable
    stat_field = "MEDIAN" if stat_type == "MEDIAN" else "MEAN"

    # --- Join & Calculate into final field ---
    try:
        arcpy.JoinField_management(fc, "HUC12", out_table, "HUC12", stat_field)
        arcpy.CalculateField_management(fc, field_name, f"!{stat_field}!")
        arcpy.DeleteField_management(fc, stat_field)
        print(f"✅ Populated {field_name}")
    except Exception:
        print(f"❌ Join/Calculate failed for {field_name}")
        traceback.print_exc()

print("\n🎉 COMPLETE — check attribute table")





Joining the HUC12 climate (and other) attribute fields to my NHDPlusV2 study area catchments layer ("") > this join/calculate field approach didn't work > needed to pivot to a cursor-based update approach (see below)

In [ ]:
import arcpy

nhd_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\CatchmentsNHDPlus_Oct3_Albers_HUCAssigned"
huc_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\Catchments_All_Upstream"

# Make layer so join works correctly
arcpy.management.MakeFeatureLayer(nhd_fc, "nhd_layer")

# Join using actual field name in HUC
arcpy.management.AddJoin("nhd_layer", "HUC12_ID", huc_fc, "HUC12")

fields_to_copy = [
    "TotalDisAreaSqKm", "PerimeterKm", "Compactness",
    "Aridity_daymetws", "ElevMedian_daymetws", "InterAnnPrecipCV_daymetws",
    "PercPrecipSnow_daymetws", "PotEvapotransEra5_daymetws", "PrecDaysMean_daymetws",
    "PrecipIntensity_daymetws", "PrecipSeasInd_daymetws", "PrecMean_daymetws",
    "SimpPrecIntensity_daymetws", "SRadMean_daymetws", "SWE_April_daymetws",
    "TempJulyMean_daymetws", "TempMax_daymetws", "TempMean_daymetws",
    "TempMin_daymetws", "WinterSumPrecipRatio_daymetws"
]

# Build lookup map of joined field names
join_field_map = {f.baseName: f.name for f in arcpy.ListFields("nhd_layer")}

print("\nJoined field mapping:")
for k,v in join_field_map.items():
    print(f"{k}  -->  {v}")

# Calculate fields
for f in fields_to_copy:
    if f not in join_field_map:
        print(f"⚠️ Field missing in join, skipping: {f}")
        continue
    
    joined_name = join_field_map[f]
    print(f"Copying {f} from {joined_name}")

    arcpy.management.CalculateField(
        "nhd_layer",
        f,                       # write into original NHD field
        f"!{joined_name}!",      # source from joined HUC field
        "PYTHON3"
    )

# Remove join safely
arcpy.management.RemoveJoin("nhd_layer", "*")

print("\n✅ SUCCESS: All fields copied into NHD")






Cursor based approach to getting the raster zonal stat (+ perimeter, area, compactness) values transferred from the HUC12 zonal stats table to my master NHDPlusV2 study are dataset

In [ ]:
import arcpy

# -------------------------------------------------------------------
# USER INPUTS
# -------------------------------------------------------------------
nhd_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\CatchmentsNHDPlus_Oct3_Albers_HUCAssigned"
huc_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\Catchments_All_Upstream"

huc_key = "HUC12"
nhd_key = "HUC12_ID"

# Fields to update in NHD from HUC table
fields_to_update = [
    "TotalDisAreaSqKm", "PerimeterKm", "Compactness",
    "Aridity_daymetws",
    "ElevMedian_daymetws",
    "InterAnnPrecipCV_daymetws",
    "PercPrecipSnow_daymetws",
    "PotEvapotransEra5_daymetws",
    "PrecDaysMean_daymetws",
    "PrecipIntensity_daymetws",
    "PrecipSeasInd_daymetws",
    "PrecMean_daymetws",
    "SimpPrecIntensity_daymetws",
    "SRadMean_daymetws",
    "SWE_April_daymetws",
    "TempJulyMean_daymetws",
    "TempMax_daymetws",
    "TempMean_daymetws",
    "TempMin_daymetws",
    "WinterSumPrecipRatio_daymetws"
]

# -------------------------------------------------------------------
# Build lookup dict from HUC feature class
# -------------------------------------------------------------------
print("Building lookup dictionary from HUC catchments...")

huc_fields = [huc_key] + fields_to_update
huc_dict = {}

with arcpy.da.SearchCursor(huc_fc, huc_fields) as cur:
    for row in cur:
        huc = row[0]
        values = row[1:]
        if huc not in huc_dict:   # avoid duplicates just in case
            huc_dict[huc] = values

print(f"✅ Loaded {len(huc_dict)} HUC records into memory")

# -------------------------------------------------------------------
# Update NHD catchments
# -------------------------------------------------------------------
print("Updating NHD polygons...")

update_fields = [nhd_key] + fields_to_update
updated = 0
missing = 0

with arcpy.da.UpdateCursor(nhd_fc, update_fields) as cur:
    for row in cur:
        huc = row[0]

        if huc in huc_dict:
            for i in range(len(fields_to_update)):
                row[i+1] = huc_dict[huc][i]
            cur.updateRow(row)
            updated += 1
        else:
            missing += 1

print(f"✅ Updated {updated:,} NHD catchments")
print(f"⚠️  {missing:,} NHD catchments had no matching HUC value")
print("Done.")


Troubleshooting NHD and HUC12 field key differences (why joins are not working for many)

In [ ]:
import arcpy

nhd_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\CatchmentsNHDPlus_Oct3_Albers_HUCAssigned"
huc_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\Catchments_All_Upstream"

# field names (adjust if your NHD field is different)
nhd_huc_field = "HUC12_ID"
huc_field = "HUC12"

# collect keys
huc_keys = set()
with arcpy.da.SearchCursor(huc_fc, [huc_field]) as cur:
    for r in cur:
        v = r[0]
        if v is None:
            continue
        s = str(v).strip()
        huc_keys.add(s)

nhd_keys = set()
with arcpy.da.SearchCursor(nhd_fc, [nhd_huc_field]) as cur:
    for r in cur:
        v = r[0]
        if v is None:
            continue
        s = str(v).strip()
        nhd_keys.add(s)

print(f"HUC table: unique HUC12 count = {len(huc_keys)}")
print(f"NHD table: unique HUC12_ID count = {len(nhd_keys)}")

common = huc_keys & nhd_keys
print(f"Exact string matches = {len(common)}")

# Try relaxed matches: pad to len 12
def normalize(s):
    s = str(s).strip()
    if len(s) < 12 and s.isdigit():
        return s.zfill(12)
    return s

norm_huc = set(normalize(k) for k in huc_keys if k)
norm_nhd = set(normalize(k) for k in nhd_keys if k)
print(f"Normalized matches (zfilled) = {len(norm_huc & norm_nhd)}")

# Show some example unmatched NHD HUCs (up to 20)
unmatched_nhd = sorted(nhd_keys - huc_keys)
print("\nExample unmatched NHD HUC12_IDs (first 20):")
for k in unmatched_nhd[:20]:
    print(" ", k)

# And example HUC keys that never matched an NHD
unmatched_huc = sorted(huc_keys - nhd_keys)
print("\nExample HUCs that don't match any NHD (first 20):")
for k in unmatched_huc[:20]:
    print(" ", k)


Script to identify 18 HUC12s not matching between datasets

In [ ]:
import arcpy

nhd_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\CatchmentsNHDPlus_Oct3_Albers_HUCAssigned"
huc_fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\Catchments_All_Upstream"

# Load unique HUC lists
nhd_hucs = {row[0] for row in arcpy.da.SearchCursor(nhd_fc, "HUC12_ID")}
huc_table_hucs = {row[0] for row in arcpy.da.SearchCursor(huc_fc, "huc12")}

# Normalize to strings
nhd_hucs = {str(h) for h in nhd_hucs}
huc_table_hucs = {str(h) for h in huc_table_hucs}

# Identify missing HUCs
unmatched = sorted(nhd_hucs - huc_table_hucs)

print(f"Missing HUC12s ({len(unmatched)} total):")
for h in unmatched:
    print(h)



Confirm gdb exists and fields weren't added to FC yet for climatic raster zonal stats

In [ ]:
import arcpy
fc = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\Catchments_All_Upstream"
print("Exists:", arcpy.Exists(fc))

fields = [f.name for f in arcpy.ListFields(fc)]
print("Number of fields:", len(fields))
for f in fields:
    print(f)


First cell of sequence that corrects mistake in first round of not differentiating between mainstem/nonmainstem sources in how values are accumulated for 17 climate rasters. This cell will identify mainstem comids within each HUC, add a field inventorying them (1 = mainstem, 0 = not) so that I can look at the map and verify. 

In [ ]:
import arcpy
from collections import Counter

# --- Inputs ---
gdb = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb"
catchments_fc = f"{gdb}\\CatchmentsNHDPlus_Oct3_Albers_HUCAssigned"

huc_field = "HUC12"
order_field = "streamorde"
levelpath_field = "levelpathi"
is_mainstem_field = "IsMainstem"

arcpy.env.workspace = gdb

# --- Add IsMainstem field if missing ---
fields = [f.name for f in arcpy.ListFields(catchments_fc)]
if is_mainstem_field not in fields:
    arcpy.AddField_management(catchments_fc, is_mainstem_field, "SHORT")
    print(f"Added field {is_mainstem_field}")

# --- Reset field to 0 ---
arcpy.CalculateField_management(catchments_fc, is_mainstem_field, 0)

# --- Build HUC12 dictionary ---
huc_dict = {}

with arcpy.da.SearchCursor(catchments_fc, [huc_field, order_field, levelpath_field]) as cursor:
    for huc, order, lvl in cursor:
        if huc not in huc_dict:
            huc_dict[huc] = []
        huc_dict[huc].append((order, lvl))

# --- Determine mainstem LevelPathI per HUC12 ---
mainstem_dict = {}

for huc, recs in huc_dict.items():

    # Find highest stream order
    max_order = max(r[0] for r in recs)

    # Filter to those reaches
    max_order_lvls = [lvl for order, lvl in recs if order == max_order]

    # Count frequency of LevelPathI among max-order reaches
    lvl_counts = Counter(max_order_lvls)
    dominant_lvl = lvl_counts.most_common(1)[0][0]

    # Store chosen levelpath for this HUC
    mainstem_dict[huc] = dominant_lvl

print("Identified dominant LevelPathI for all HUC12s.")

# --- Update IsMainstem for chosen LevelPathI within each HUC ---
with arcpy.da.UpdateCursor(catchments_fc, [huc_field, levelpath_field, is_mainstem_field]) as cursor:
    for row in cursor:
        huc, lvl = row[0], row[1]
        if huc in mainstem_dict and lvl == mainstem_dict[huc]:
            row[2] = 1
        else:
            row[2] = 0
        cursor.updateRow(row)

print("✅ Mainstem identification complete — IsMainstem field populated.")



Part 2 of 2 Script - Updating the zonal stat fields (climatic rasters) in "CatchmentsNHDPlus_Oct3_Albers_HUCAssigned" for non-mainstem comids to have local HUC12 zone values only, not the entire upstream accumulation from the mainstem

In [ ]:
import arcpy

# ---------------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------------
gdb = r"C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb"

nhd_fc = f"{gdb}\\CatchmentsNHDPlus_Oct3_Albers_HUCAssigned"
huc_fc = f"{gdb}\\ZonalChFeature_HUC12_EntireStudyArea_cleaned"

huc_field = "HUC12"
mainstem_field = "IsMainstem"

# 17 raster zonal stat fields:
raster_fields = [
    "Aridity_daymetws",
    "ElevMedian_daymetws",
    "InterAnnPrecipCV_daymetws",
    "PercPrecipSnow_daymetws",
    "PotEvapotransEra5_daymetws",
    "PrecDaysMean_daymetws",
    "PrecipIntensity_daymetws",
    "PrecipSeasInd_daymetws",
    "PrecMean_daymetws",
    "SimpPrecIntensity_daymetws",
    "SRadMean_daymetws",
    "SWE_April_daymetws",
    "TempJulyMean_daymetws",
    "TempMax_daymetws",
    "TempMean_daymetws",
    "TempMin_daymetws",
    "WinterSumPrecipRatio_daymetws"
]

arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

print("Building lookup dictionary from HUC12 zonal stats...")

# ---------------------------------------------------------------------
# STEP 1 — Build dictionary: HUC12 → { field:value, field:value, ... }
# ---------------------------------------------------------------------
huc_stats = {}

fields_to_read = [huc_field] + raster_fields

with arcpy.da.SearchCursor(huc_fc, fields_to_read) as cur:
    for row in cur:
        huc = row[0]
        stats = row[1:]
        huc_stats[huc] = stats

print(f"Loaded zonal stats for {len(huc_stats)} HUC12 polygons.")


# ---------------------------------------------------------------------
# STEP 2 — Update NHD catchments where IsMainstem == 0
# ---------------------------------------------------------------------
update_fields = [huc_field, mainstem_field] + raster_fields

print("Updating non-mainstem catchments...")

count_updated = 0
count_skipped = 0

with arcpy.da.UpdateCursor(nhd_fc, update_fields) as cur:
    for row in cur:
        huc = row[0]
        is_main = row[1]

        # Skip mainstem reaches entirely
        if is_main == 1:
            count_skipped += 1
            continue

        if huc not in huc_stats:
            # This can happen for closed basins or mismatched HUCs
            continue

        # Insert the HUC-level zonal stats into the corresponding fields
        # row structure: [HUC12, IsMainstem, f1, f2, f3, ...]
        huc_values = huc_stats[huc]

        for i in range(len(raster_fields)):
            row[2 + i] = huc_values[i]

        cur.updateRow(row)
        count_updated += 1

print(f"\n🎉 Zonal stats update complete!")
print(f"    Updated non-mainstem catchments: {count_updated}")
print(f"    Skipped mainstem catchments:     {count_skipped}")
